# FAA-CxER-no-groundtruth — exploration notebook

A thin companion to the CLI scripts in `scripts/`. Run
`python scripts/01_run_pipeline.py` at least once (needs a GPU) so
`data/analysis_results.json` exists before running the cells below —
or just inspect the normalizer and rule checker directly, which need no
GPU or model at all.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from faa_cxer_ng.config import default_config
from faa_cxer_ng.io_utils import load_json

cfg = default_config()
cfg

## No GPU needed: the normalizer resolving ICAO verbalization differences

This is the piece that stops the rule checker from flagging "United 123"
vs "United one two three" as a false error.

In [ ]:
from faa_cxer_ng.normalization import ATCValueNormalizer

norm = ATCValueNormalizer()

print(norm.values_equivalent("United 123", "United one two three", "callsign"))   # True — same callsign, different wording
print(norm.values_equivalent("Delta 345", "Delta 3456", "callsign"))               # False — genuinely different digits
print(norm.values_equivalent("two seven left", "27L", "runway"))                   # True
print(norm.values_equivalent("six thousand feet", "6000 feet", "altitude"))        # True
print(norm.values_equivalent("descend to the assigned altitude", "6000 feet", "altitude"))  # None — uncertain, not a guess

## No GPU needed: the rule checker on a synthetic example

Builds a `ConversationContext` by hand (skipping stage 1) to show what the
rule-based layer catches with zero LLM calls.

In [ ]:
from faa_cxer_ng.conversation import Turn
from faa_cxer_ng.rule_checker import RuleBasedChecker
from faa_cxer_ng.schemas import CallsignMention, ConversationContext, CriticalInstruction, Readback

ctx = ConversationContext(
    primary_callsign="Delta 345",
    total_turns=4, controller_turns=2, pilot_turns=2,
    callsign_mentions=[
        CallsignMention(turn_index=0, speaker="controller", callsign_text="Delta 345"),
        CallsignMention(turn_index=2, speaker="controller", callsign_text="Delta 3456"),
    ],
    critical_instructions=[
        CriticalInstruction(turn_index=0, instruction_type="runway", raw_instruction="cleared to land runway 27L",
                            key_values={"runway": "27L"}),
    ],
    actual_readbacks=[
        Readback(turn_index=1, readback_for_turn=0, readback_for_type="runway",
                pilot_text="cleared to land runway two seven right", values_extracted={"runway": "two seven right"}),
    ],
)
turns = [Turn(0, "controller", "..."), Turn(1, "pilot", "..."), Turn(2, "controller", "..."), Turn(3, "pilot", "...")]

errors, uncertain = RuleBasedChecker().check(turns, ctx)
for e in errors:
    print(f"[{e['error_type']}] turn {e['turn_index']}: {e['actual_behavior']}")

## With a completed run: corpus-level summary

In [ ]:
results = load_json(cfg.results_path)
print(f"{len(results)} conversations analyzed")

from faa_cxer_ng.aggregate import ReportAggregator
agg = ReportAggregator(results)
agg.print_summary()

## Next steps

- `scripts/01_run_pipeline.py` — run the full pipeline (needs a GPU + vLLM)
- `scripts/02_summarize_results.py` — corpus-level stats and plots (no GPU needed)

See the top-level README.md for the full architecture explanation.